# Virtual Environments & Packages
**Topic:** Python Fundamentals

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** why virtual environments are necessary and what problem they solve
- **Describe** the difference between conda environments and venv/pip environments
- **Interpret** a `requirements.txt` or `environment.yml` file as a reproducibility contract

---
## How we got here

In *11: Debugging & Reading Tracebacks* we learned to diagnose errors in running code. Some of the most confusing errors in data science come not from your code but from your environment: the wrong version of pandas, a missing package, or a conflict between two libraries. This notebook explains the infrastructure that prevents those problems.

---
## Why this matters for data science

Reproducibility is a core requirement of professional data science. A model trained with `scikit-learn==1.3.0` may produce different results with `scikit-learn==1.5.0` due to changes in default hyperparameters. Sharing a project without a pinned environment file is handing someone a recipe without specifying ingredient brands. Every serious data science project starts with creating a virtual environment and ends with exporting it.

---
## Try it yourself

In [2]:
# ▶ Run this cell and observe the output.
# Then try changing the values and running again.

# These are terminal commands — not Python code.
# Read them as a recipe, then try running them in your terminal.

workflow = [
    ('Create environment', 'conda create -n demo_env python=3.12 -y'),
    ('Activate',           'conda activate demo_env'),
    ('Install packages',   'pip install pandas numpy scikit-learn'),
    ('Check installed',    'pip list'),
    ('Deactivate',         'conda deactivate'),
]

for step, cmd in workflow:
    print(f'{step:<25} →  {cmd}')

Create environment        →  conda create -n demo_env python=3.12 -y
Activate                  →  conda activate demo_env
Install packages          →  pip install pandas numpy scikit-learn
Check installed           →  pip list
Deactivate                →  conda deactivate


In [ ]:
# ✏️ Your turn — modify this code:
# 1. Change 'data_science_environment' to any name you prefer
# 2. Add 'matplotlib' to the pip install command
# 3. What flag does pip install --upgrade use? Why is it useful?

env_name = 'data_science_environment'

commands = {
    'create':   f'conda create -n {env_name} python=3.12 -y',
    'activate': f'conda activate {env_name}',
    'install':  'pip install pandas numpy scikit-learn jupyter',
    'export':   f'conda env export > {env_name}.yml',
}

for action, cmd in commands.items():
    print(f'{action}: {cmd}')

In [ ]:
# 🎯 Challenge:
# Write the sequence of terminal commands (as a Python list of strings) to:
#   1. Create a new conda environment called 'my_project' with Python 3.12
#   2. Activate it
#   3. Install pandas and numpy
#   4. Save the environment to a yml file called 'my_project.yml'
# Hint: conda env export > filename.yml saves an environment

commands = [
    # Your commands here (each as a string)
]

---
## What's happening?

A virtual environment is an isolated directory containing its own Python interpreter and installed packages. Activating an environment redirects `python` and `pip` to that directory, so package installs do not affect other environments or the system Python.

| Tool | Creates environments with | Package manager | Best for |
|------|--------------------------|----------------|----------|
| `venv` | `python -m venv env/` | `pip` | Pure Python projects |
| `conda` | `conda create -n myenv python=3.12` | `conda` + `pip` | Data science (handles non-Python deps like BLAS) |
| `poetry` | `poetry new myproject` | `poetry` | Projects with complex dependency resolution |

```bash
# Creating and activating a conda environment
conda create -n ds_project python=3.12
conda activate ds_project
pip install pandas scikit-learn plotly ipywidgets

# Export for reproducibility
pip freeze > requirements.txt
# or for conda:
conda env export > environment.yml
```

### The three files that make a project reproducible

`requirements.txt` lists exact package versions (`pandas==2.1.4`). `environment.yml` does the same for conda and includes the Python version and conda channels. `pyproject.toml` (poetry/modern standard) separates pinned versions for development from minimum versions for distribution.

Return to the diagram widget and trace what happens when a collaborator clones the repo and runs `conda env create -f environment.yml`, this is the hand-off that reproducibility makes possible.

---
## Real-world example: Package version conflicts in a team data science project

The chart shows a realistic dependency conflict scenario: two data scientists on the same team installed different versions of scikit-learn into their shared system Python. The model trained by Data Scientist A produces a different default regularization strength than the model trained by Data Scientist B, even though they ran the same code.

Notice:

- **Notice:** The bars represent the model's test accuracy under each environment; the difference is small but real, and it would be invisible without version control on the environment itself
- **Notice:** The conflict is not a bug in either version of scikit-learn; it is a legitimate behavioral change between releases that the release notes document but teams rarely check
- **Notice:** Pinning to `scikit-learn==1.3.2` in `requirements.txt` makes both bars identical, which is the entire point of environment management

> **Discussion question:** Your manager asks why you need a virtual environment per project rather than just one shared data science environment for the whole team. How would you explain the risk of not doing this using the scenario in the chart above?

In [ ]:
# ── Package version impact on model accuracy (simulated) ──────────────────────
environments = [
    "DS-A: scikit-learn 1.2.2\n(no env, system Python)",
    "DS-B: scikit-learn 1.5.1\n(no env, system Python)",
    "Pinned: scikit-learn 1.3.2\n(shared environment.yml)",
    "Pinned: scikit-learn 1.3.2\n(shared environment.yml)",
]
test_accuracy = [0.847, 0.831, 0.843, 0.843]
colors = ["#E45756", "#E45756", "#55A868", "#55A868"]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(range(len(environments)), test_accuracy, color=colors)
for bar, acc in zip(bars, test_accuracy):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f"{acc:.1%}", ha="center", va="bottom", fontsize=11)
ax.set_title(
    "Same Code, Different Environments — Model Accuracy Varies by scikit-learn Version",
    fontsize=12)
ax.set_xlabel("Developer Environment")
ax.set_ylabel("Test Accuracy")
ax.set_xticks(range(len(environments)))
ax.set_xticklabels(environments, fontsize=9)
ax.set_ylim(0.80, 0.87)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


### Environment management quick reference

| Task | conda command | pip / venv command |
|------|--------------|-------------------|
| Create environment | `conda create -n name python=3.12` | `python -m venv env/` |
| Activate | `conda activate name` | `source env/bin/activate` (Mac/Linux) |
| Install a package | `conda install pandas` or `pip install pandas` | `pip install pandas` |
| Export environment | `conda env export > environment.yml` | `pip freeze > requirements.txt` |
| Recreate from file | `conda env create -f environment.yml` | `pip install -r requirements.txt` |
| List environments | `conda env list` | `ls` the directory where you keep envs |

---
## Key takeaway

> **A virtual environment is an isolated box containing a specific Python version and pinned packages; exporting it to a file is the contract that lets a teammate or a server reproduce your results exactly.**

---
*Next up: NumPy — the array library that powers pandas, scikit-learn, and every numerical operation in your data science stack*